In [0]:

sqlserver_host = "your-sql-server-database.windows.net"
sqlserver_port = "1433"
sqlserver_database = "your_database"
sqlserver_user = "your_username"
sqlserver_password = "your_password"

secret_scope_name = "banking-scope"


secret_key_name = "sqlserver-connection-json"



In [0]:
import json

connection_config = {
    "host": sqlserver_host,
    "port": sqlserver_port,
    "database": sqlserver_database,
    "user": "*" ,
    "password": "*",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


connection_json = json.dumps(connection_config)
print("Generated JSON Configuration:")
print(connection_json)

In [0]:
# Retrieve the Databricks notebook context object
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

# Extract the Databricks workspace API URL from the context
api_url = ctx.apiUrl().getOrElse(None)

# Extract the Databricks API token from the context
api_token = ctx.apiToken().getOrElse(None)

print(api_url)
print(api_token)


import requests
import json


DATABRICKS_INSTANCE = api_url
DATABRICKS_TOKEN = api_token


scope_name = secret_scope_name
backend_type = "DATABRICKS"

In [0]:
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"
# print(url)
# The URL for creating a secret scope in Databricks REST API is constructed here:
# {DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create

headers = {
     "Authorization": f"Bearer {DATABRICKS_TOKEN}",
     "Content-Type": "application/json"
 }

payload = {
     "scope": scope_name
 }

response = requests.post(url, headers=headers, data=json.dumps(payload))

if response.status_code == 200:
     print (f"Secret scope '{scope_name}' created successfully.")
else:
     print("Failed to create secret scope")
     print("Status Code:", response.status_code)
     print("Response:", response.text)

In [0]:
import requests
import json


scope = secret_scope_name
secret_key = secret_key_name
secret_value = connection_json


url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"


headers = {
    "Authorization" : f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope":  scope,
    "key": secret_key,
    "string_value": secret_value
}

response = requests.post(url, headers=headers, data = json.dumps(payload))


if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'")
else:
    print("Failed to create secret.")
    print("Status : " ,response.status_code)
    print("Response:", response.text)

In [0]:
try:
    retrieved_json = dbutils.secrets.get(
        scope = secret_scope_name,
        key= secret_key_name
    )
    print("Secret retrieved_json.")

    parsed = json.loads(retrieved_json)
    print ("parsed JSON: ")
    print(parsed)

except Exception as e:
    print("Secret verification failed:")
    print("str(e)")